# Supplementary Results 6 — Variant-level pleiotropy modelling

The variant-by-disease effect matrix, directional concordance, and the two-component mixture of
squared effect sizes at highly pleiotropic variants.

Numbers are written to `results/sr06_variant_pleiotropy.json`.

**Provenance.** `~/Projects/EGL_and_training_set/archive/gentropy_paper/24_real_ps.ipynb`,
cells 26-57: the two deduplication passes that build the matrix, the `> 10` disease cut, and the
`GaussianMixture(random_state=123)` fit on squared effect sizes with a 10-point minimum. Ported
here with those definitions unchanged. The four supplementary figures of this section
(SR2, SR3, SR4, SR6) are not rebuilt; SR6's counts are checked below.

**The published 1,403 diseases is the count before deduplication.** The archive counts diseases on
its input table (cell 29, 77,405 rows) and the matrix on the output of two dedupe passes (cell 32,
54,728 rows). The first pass keeps one row per variant and study, so a study carrying several
disease terms contributes only one of them, and 95 diseases disappear: the matrix spans **1,308**
diseases, not 1,403. Both are computed below, and 1,403 is the registered value because the
manuscript's own parenthetical defines it that way — "1,403 diseases (each with at least one
associated credible set)".

In [1]:
import numpy as np
import pandas as pd
from gentropy.common.session import Session
from pyspark.sql import Window
from pyspark.sql import functions as f
from sklearn.mixture import GaussianMixture

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
numbers = {}

MIN_POINTS = 10  # a mixture is fitted only where a variant has at least this many effect estimates
RANDOM_STATE = 123

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/22 16:10:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## The variant-by-disease effect matrix

Qualifying disease credible sets, exploded to one row per disease term. Two deduplication passes,
in the archive's order: keep the largest absolute rescaled effect per variant and study, then the
largest per variant and disease. Non-significant associations are absent rather than zero.

In [2]:
qualifying = session.spark.read.parquet(paper.derived("qualifying_credible_sets")).select("studyLocusId")
rows = (
    session.spark.read.parquet(paper.derived("lead_variant_effect"))
    .join(qualifying, "studyLocusId", "inner")
    .select(
        "variantId",
        "studyId",
        f.explode("diseaseIds").alias("diseaseId"),
        f.col("rescaledStatistics.absEstimatedBeta").alias("absEstimatedBeta"),
        f.col("majorLdPopulationMaf.value").alias("maf"),
        # carried through the dedupes so concordance can be computed on the matrix itself
        f.signum(f.col("rescaledStatistics.minorAlleleEstimatedBeta")).alias("sign"),
    )
    .cache()
)

per_study = Window.partitionBy("variantId", "studyId").orderBy(f.desc("absEstimatedBeta"))
per_disease = Window.partitionBy("diseaseId", "variantId").orderBy(f.desc("absEstimatedBeta"))
deduplicated = rows.withColumn("rank", f.row_number().over(per_study)).filter(f.col("rank") == 1).drop("rank").cache()
matrix = (
    deduplicated.withColumn("rank", f.row_number().over(per_disease)).filter(f.col("rank") == 1).drop("rank").cache()
)

numbers["S6.01"] = matrix.select("variantId").distinct().count()
# The registered disease count is the one the manuscript's parenthetical describes: diseases with at
# least one associated qualifying credible set, before the deduplication. Deduplicating by variant
# and study drops the 95 diseases that only ever appear alongside another term in the same study.
numbers["S6.02"] = rows.select("diseaseId").distinct().count()
in_matrix = matrix.select("diseaseId").distinct().count()
print(f"input rows: {rows.count():,} | diseases with a credible set: {numbers['S6.02']:,}")
print(f"matrix rows: {matrix.count():,} | variants: {numbers['S6.01']:,} | diseases in the matrix: {in_matrix:,}")

# The effect matrix keeps largest-|beta| per disease-variant pair, so these must not move either.
assert (numbers["S6.01"], numbers["S6.02"], in_matrix) == (40706, 1403, 1308)

input rows: 77,405 | diseases with a credible set: 1,403


matrix rows: 54,728 | variants: 40,706 | diseases in the matrix: 1,308


## How many diseases each variant is associated with

In [3]:
features = pd.read_parquet(
    paper.derived("variant_features"), columns=["variantId", "uniqueDiseases", "betaSignConcordance"]
)
pleiotropic = features["uniqueDiseases"] > 1

numbers["S6.03"] = int(pleiotropic.sum())
numbers["S6.04"] = round(100 * pleiotropic.mean(), 0)
numbers["S6.05"] = round(float(features["uniqueDiseases"].mean()), 2)
numbers["S6.06"] = int(features["uniqueDiseases"].max())
print({k: numbers[k] for k in ["S6.03", "S6.04", "S6.05", "S6.06"]})

# vPS keeps counting every disease term: these are the values the unchanged pipeline produced.
assert (numbers["S6.03"], numbers["S6.04"], numbers["S6.05"], numbers["S6.06"]) == (9828, 24.0, 1.48, 85)

{'S6.03': 9828, 'S6.04': np.float64(24.0), 'S6.05': 1.48, 'S6.06': 85}


## Directional concordance

Concordance is the largest share of same-direction effects for a variant.

Four readings are computed. `betaSignConcordance` is the published column: every credible set of the
variant, direction taken from `rescaledStatistics.minorAlleleEstimatedBeta`. The matrix reading takes
one signed effect per variant and disease off the effect matrix above. `leadVPS` is the first
redefinition: only the credible sets of studies mapped to exactly one disease term contribute, and
direction comes from the harmonised effect-allele beta with no minor-allele conversion.
`signedLeadVPS` is the amendment: a credible set whose `directionOfEffect` is null carries no
directional information and so does not contribute at all, which means every counted disease is
signed and concordance is computable wherever `signedLeadVPS` is at least 1.

S6.07 and S6.08 are registered on the amendment. Under it there is no undefined group inside the
universe — a variant either has a signed contributing credible set or is excluded outright — so the
percentage no longer depends on which denominator is chosen. The manuscript reports 1,793 of the
9,828 pleiotropic variants below 1, which no reading reproduces and no surviving notebook computes.

In [4]:
# Every reading of "concordance < 1", since the published 1,793 matches none of them and no
# surviving notebook computes it. `betaSignConcordance` is the pipeline's own column, computed over
# a variant's credible sets; the matrix reading takes one signed effect per variant and disease,
# which is the only reading under which a non-pleiotropic variant is 1 by construction.
from_column = int(((features["uniqueDiseases"] > 1) & (features["betaSignConcordance"] < 1)).sum())
all_variants = int((features["betaSignConcordance"] < 1).sum())

signed = (
    matrix.filter(f.col("sign") != 0)
    .groupBy("variantId")
    .agg(f.count("*").alias("n"), f.sum(f.when(f.col("sign") > 0, 1).otherwise(0)).alias("positive"))
    .toPandas()
)
signed["concordance"] = np.maximum(signed["positive"], signed["n"] - signed["positive"]) / signed["n"]
multi_disease = signed[signed["n"] > 1]
from_matrix = int((multi_disease["concordance"] < 1).sum())

readings = pd.DataFrame(
    [
        {
            "reading": "variant_features column, the 9,828 pleiotropic variants",
            "below 1": from_column,
            "universe": numbers["S6.03"],
        },
        {"reading": "variant_features column, every variant", "below 1": all_variants, "universe": len(features)},
        {
            "reading": "matrix effects, the 9,828 pleiotropic variants",
            "below 1": int(
                (
                    signed[signed["variantId"].isin(features.loc[features["uniqueDiseases"] > 1, "variantId"])][
                        "concordance"
                    ]
                    < 1
                ).sum()
            ),
            "universe": numbers["S6.03"],
        },
        {
            "reading": "matrix effects, variants with more than one disease in the matrix",
            "below 1": from_matrix,
            "universe": len(multi_disease),
        },
    ]
)
readings["%"] = (100 * readings["below 1"] / readings["universe"]).round(1)

# The redefinition. The universe is the variants whose lead_vPS exceeds 1; those with fewer than
# two signed diseases have no concordance and are counted on their own, never merged into either
# group. Variants with no contributing credible set have no lead_vPS at all.
redefined = pd.read_parquet(
    paper.derived("variant_features"),
    columns=[
        "variantId",
        "leadVPS",
        "leadVPSDefined",
        "leadDirectionalConcordance",
        "leadConcordanceDefined",
        "signedLeadVPS",
        "signedLeadVPSDefined",
        "signedLeadDirectionalConcordance",
    ],
)
lead_pleiotropic = redefined[redefined["leadVPSDefined"] & (redefined["leadVPS"] > 1)]
lead_defined = lead_pleiotropic[lead_pleiotropic["leadConcordanceDefined"]]
lead_below_one = int((lead_defined["leadDirectionalConcordance"] < 1).sum())

# The amendment. Every counted disease is signed, so the universe and the defined set coincide.
gated_pleiotropic = redefined[redefined["signedLeadVPSDefined"] & (redefined["signedLeadVPS"] > 1)]
assert gated_pleiotropic["signedLeadDirectionalConcordance"].notna().all()
gated_below_one = int((gated_pleiotropic["signedLeadDirectionalConcordance"] < 1).sum())

numbers["S6.07"] = gated_below_one
numbers["S6.08"] = round(100 * gated_below_one / len(gated_pleiotropic), 0)
print("published: 1,793 of 9,828 (18%)")
print(
    "redefinition:",
    {
        "lead_vPS undefined (no contributing credible set)": int((~redefined["leadVPSDefined"]).sum()),
        "lead_vPS > 1": len(lead_pleiotropic),
        "concordance defined": len(lead_defined),
        "concordance undefined (< 2 signed diseases)": len(lead_pleiotropic) - len(lead_defined),
        "defined and concordant": int((lead_defined["leadDirectionalConcordance"] == 1).sum()),
        "defined and discordant": lead_below_one,
        "% discordant of the defined": round(100 * lead_below_one / len(lead_defined), 1),
        "% discordant of lead_vPS > 1": round(100 * lead_below_one / len(lead_pleiotropic), 1),
    },
)
print(
    "amendment (sign-gated):",
    {
        "excluded (no signed contributing credible set)": int((~redefined["signedLeadVPSDefined"]).sum()),
        "signedLeadVPS > 1": len(gated_pleiotropic),
        "concordance undefined": 0,
        "concordant": len(gated_pleiotropic) - gated_below_one,
        "discordant": gated_below_one,
        "% discordant": round(100 * gated_below_one / len(gated_pleiotropic), 1),
    },
)
readings

published: 1,793 of 9,828 (18%)
redefinition: {'lead_vPS undefined (no contributing credible set)': 2433, 'lead_vPS > 1': 6383, 'concordance defined': 5777, 'concordance undefined (< 2 signed diseases)': 606, 'defined and concordant': 4751, 'defined and discordant': 1026, '% discordant of the defined': 17.8, '% discordant of lead_vPS > 1': 16.1}
amendment (sign-gated): {'excluded (no signed contributing credible set)': 5234, 'signedLeadVPS > 1': 5919, 'concordance undefined': 0, 'concordant': 4868, 'discordant': 1051, '% discordant': 17.8}


,reading,below 1,universe,%
0,"variant_features column, the 9,828 pleiotropic...",1415,9828,14.4
1,"variant_features column, every variant",1652,40706,4.1
2,"matrix effects, the 9,828 pleiotropic variants",1260,9828,12.8
3,"matrix effects, variants with more than one di...",1260,6494,19.4


## Mixture of squared effect sizes at highly pleiotropic variants

Variants associated with **more than** 10 diseases, fitted where at least 10 effect estimates
survive. One- and two-component Gaussian mixtures on the squared absolute effects are compared by
BIC.

In [5]:
counts = deduplicated.groupBy("variantId").agg(f.countDistinct("diseaseId").alias("uniqueDiseases"))
highly = counts.filter(f.col("uniqueDiseases") > MIN_POINTS).select("variantId")
effects = matrix.join(highly, "variantId", "inner").select("variantId", "absEstimatedBeta").toPandas()
print(f"effect estimates at highly pleiotropic variants: {len(effects):,}")


def fit_mixture(values):
    """One- and two-component mixtures of squared effects, as in the archive notebook."""
    squared = values.dropna().to_numpy() ** 2
    if len(squared) < MIN_POINTS:
        return None
    reshaped = squared.reshape(-1, 1)
    one = GaussianMixture(n_components=1, random_state=RANDOM_STATE).fit(reshaped)
    two = GaussianMixture(n_components=2, random_state=RANDOM_STATE).fit(reshaped)
    assignment = two.predict(reshaped)
    share = float((assignment == 1).mean())
    return {
        "points": len(squared),
        "bic1": one.bic(reshaped),
        "bic2": two.bic(reshaped),
        "clusterBalance": min(share, 1 - share),
        "mean1": float(two.means_[0][0]),
        "mean2": float(two.means_[1][0]),
    }


fits = pd.DataFrame(
    [
        {"variantId": variant, **fit}
        for variant, values in effects.groupby("variantId")["absEstimatedBeta"]
        if (fit := fit_mixture(values)) is not None
    ]
)
print(f"variants fitted: {len(fits):,}")

effect estimates at highly pleiotropic variants: 2,153


variants fitted: 118


In [6]:
two_component = fits["bic2"] < fits["bic1"]
ratios = np.maximum(fits["mean1"] / fits["mean2"], fits["mean2"] / fits["mean1"])

numbers["S6.09"] = len(fits)
numbers["S6.10"] = int(two_component.sum())
numbers["S6.11"] = round(100 * two_component.mean(), 0)
numbers["S6.12"] = round(float(ratios.mean()), 1)
numbers["S6.13"] = round(float(ratios.median()), 1)
numbers["S6.14"] = round(100 * float(fits["clusterBalance"].mean()), 0)
print({k: numbers[k] for k in ["S6.09", "S6.10", "S6.11", "S6.12", "S6.13", "S6.14"]})

{'S6.09': 118, 'S6.10': 100, 'S6.11': np.float64(85.0), 'S6.12': 14.5, 'S6.13': 7.7, 'S6.14': 22.0}


## Supplementary Figure SR6 — therapeutic areas against diseases per cluster

The caption's counts, over the colocalisation clusters of Results 4.

In [7]:
cluster_table = pd.read_parquet(paper.derived("variant_clusters"))
coordinates = cluster_table.groupby(["uniqueDiseases", "uniqueTherapeuticAreas"]).size()

numbers["S6.15"] = len(cluster_table)
numbers["S6.16"] = int(len(coordinates))
numbers["S6.17"] = int(coordinates.get((1, 1), 0))
print(
    f"clusters {numbers['S6.15']:,} | distinct coordinates {numbers['S6.16']} | "
    f"at one disease and one therapeutic area {numbers['S6.17']:,}"
)
coordinates.sort_values(ascending=False).head()

clusters 20,041 | distinct coordinates 226 | at one disease and one therapeutic area 13,424


uniqueDiseases  uniqueTherapeuticAreas
1               1                         13424
2               2                          1657
                1                          1414
3               2                           687
                3                           344
dtype: int64

## Write the results

In [8]:
print(paper.save_results("sr06_variant_pleiotropy", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr06_variant_pleiotropy.json


,computed
S6.01,40706.00
S6.02,1403.00
S6.03,9828.00
S6.04,24.00
S6.05,1.48
S6.06,85.00
S6.07,1051.00
S6.08,18.00
S6.09,118.00
S6.10,100.00
